# 02. Data Quality, Hygiene & Contract Auditing

## Methodological Framing
Real search performance datasets frequently exhibit zero-inflation, reporting latency, sporadic tracking gaps, and sparse query coverage. This notebook implements formal data hygiene checks to establish quality boundaries before any modeling occurs.

### Objectives:
1. Audit null value prevalence across all candidate attributes.
2. Profile zero-inflation in click and impression counts.
3. Enforce contract rules regarding date continuity and temporal identifiers.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

root_dir = Path.cwd().parent if Path.cwd().name == "work" else Path.cwd()
sys.path.insert(0, str(root_dir))

from src.config import load_config
from src.data import load_dataset

config = load_config()
try:
    df = load_dataset(config.raw_data_path, config)
    print(f"Connected: {len(df):,} records.")
except FileNotFoundError:
    print("[STATUS: AWAITING REAL DATA EXECUTION] - Dataset connection required.")
    df = None

## Step 1: Missingness Audit

In [ ]:
if df is not None:
    null_counts = df.isnull().sum()
    null_pct = (null_counts / len(df) * 100).round(2)
    missing_summary = pd.DataFrame({"Missing Count": null_counts, "Missing Pct (%)": null_pct})
    display(missing_summary[missing_summary["Missing Count"] > 0])
else:
    print("Connect warehouse dataset to compute missingness statistics.")

## Step 2: Zero-Inflation and Cardinality Audit

In [ ]:
if df is not None:
    for metric in ["clicks", "impressions"]:
        if metric in df.columns:
            zero_share = (df[metric] == 0).mean() * 100
            print(f"{metric} zero-rate: {zero_share:.2f}%")
else:
    print("Connect warehouse dataset to run zero-inflation diagnostics.")